---


### **作業上繳設定**

1. **檔案 → 在雲端硬碟中儲存副本**
2. 對老師分享的 `2026檢定培訓營` 按 **「新增雲端硬碟捷徑」** 加到「我的雲端硬碟」（課堂會引導）
3. 確認老師給的是 **編輯者** 權限（否則無法上傳）
4. 填好下方 **DAY**（Day1、Day2…）與 **STUDENT_NAME**（你的姓名）
5. 依序執行 3 個灰色程式格：設定 → 掛載 Drive → **上繳工具**（應顯示 `上繳工具已載入（2026-06-15）`）
6. 每題流程：**寫程式 → 執行作答格 → 執行 `make_submit_button("a001")` 格 → 按上繳**

上繳後檔案會出現在：

```
2026檢定培訓營 / Day1 / 王小明 / a001.py
```

> `Day1`、姓名等子資料夾**不用事先建立**，第一次上繳會自動產生。
> 若改過答案，要**再執行一次** `make_submit_button(...)` 那格，再按上繳。


In [ ]:
# 老師分享的共用資料夾（捷徑到「我的雲端硬碟」後的路徑，通常不用改）
SUBMIT_BASE = "/content/drive/MyDrive/橘子蘋果/APCS/2026檢定培訓營"

# ===== 請填入 =====
DAY = "Day1"           # 統一格式：Day1、Day2、Day3...
STUDENT_NAME = "王小明"  # 你的姓名


In [ ]:
#@title 首次使用需授權 Google 帳號
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive 已掛載")


In [ ]:
#@title 上傳相關設定
import os

import ipywidgets as widgets
from IPython.display import clear_output, display
from IPython import get_ipython

_NOTEBOOK_CACHE = None
SUBMIT_TOOL_VERSION = "2026-06-15"


def _user_ns():
    ip = get_ipython()
    return ip.user_ns if ip is not None else globals()


def _resolve_submit_folder():
    ns = _user_ns()
    base = ns.get("SUBMIT_BASE", "")
    day = ns.get("DAY", "").strip()
    name = ns.get("STUDENT_NAME", "").strip()

    if not base:
        raise ValueError("SUBMIT_BASE 未設定")
    if not os.path.isdir(base):
        raise ValueError(
            f"找不到共用資料夾：{base}\n"
            "請確認已掛載 Drive，且已將「2026檢定培訓營」捷徑加到「我的雲端硬碟」"
        )
    if not day:
        raise ValueError("請填 DAY（例如 Day1）")
    if not day.startswith("Day") or not day[3:].isdigit():
        raise ValueError("DAY 請用 Day1、Day2 格式")
    if not name:
        raise ValueError("請填 STUDENT_NAME（你的姓名）")

    return os.path.join(base, day, name)


def _submit_filename(question_id):
    return f"{question_id}.py"


def _cell_source_text(cell):
    src = cell.get("source", [])
    if isinstance(src, str):
        return src
    return "".join(src)


def _capture_notebook_now():
    """在執行 cell 當下讀取 Colab 畫面上的講義（不能在按鈕 callback 裡讀）。"""
    try:
        from google.colab import _message

        reply = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
        if reply and "ipynb" in reply:
            return reply["ipynb"]
    except Exception:
        pass
    return None


def _extract_answer_code(nb, question_id):
    marker = f"#start-{question_id}"
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = _cell_source_text(cell)
        if marker not in src:
            continue
        lines = []
        for line in src.splitlines():
            stripped = line.strip()
            if stripped == marker or stripped == "#start":
                continue
            lines.append(line)
        code = "\n".join(lines).strip()
        return code + "\n" if code else ""
    raise ValueError(
        f"找不到作答區 {marker}。請確認開啟的是「_上繳」版講義，並在作答格寫好程式。"
    )


def make_submit_button(question_id):
    global _NOTEBOOK_CACHE

    nb = _capture_notebook_now()
    if nb is None:
        print("⚠️ 無法讀取講義。請在 Google Colab 中執行此格。")
        _NOTEBOOK_CACHE = None
    else:
        _NOTEBOOK_CACHE = nb
        try:
            _extract_answer_code(nb, question_id)
        except ValueError as exc:
            print(f"⚠️ {exc}")

    btn = widgets.Button(description="上繳", button_style="success", icon="cloud-upload")
    output = widgets.Output()

    def on_submit(_):
        with output:
            clear_output(wait=True)
            print("上繳中，請稍候…")
            try:
                if _NOTEBOOK_CACHE is None:
                    print("⚠️ 請先重新執行此格（make_submit_button），再按上繳。")
                    return

                if not os.path.isdir("/content/drive/MyDrive"):
                    print("⚠️ 請先執行上方的「掛載 Google Drive」cell")
                    return

                code = _extract_answer_code(_NOTEBOOK_CACHE, question_id)
                if not code.strip():
                    print("⚠️ 作答區是空的，請先寫程式，再重新執行此格，然後按上繳")
                    return

                target_dir = _resolve_submit_folder()
                os.makedirs(target_dir, exist_ok=True)

                filename = _submit_filename(question_id)
                filepath = os.path.join(target_dir, filename)

                with open(filepath, "w", encoding="utf-8") as f:
                    f.write(code)

                print("✅ 上繳成功！")
                print(f"檔案：{filename}")
                print(f"路徑：{filepath}")
            except Exception as exc:
                print(f"❌ 上繳失敗：{exc}")

    btn.on_click(on_submit)
    display(widgets.VBox([btn, output]))


print(f"上繳工具已載入（{SUBMIT_TOOL_VERSION}）")

## **ZeroJudge(3)** (60 min)



### **a001. 哈囉**
#### **題目說明：**
#### 請寫一個程式，可以讀入指定的字串，並且輸出指定的字串。
#### 比如：輸入字串 "world", 則請輸出 "hello, world"

```
Sample Input #1 : world
Sample Output #1 : hello, world
```
```
Sample Input #2 : C++
Sample Output #2 : hello, C++
```




In [ ]:
#start-a001


In [ ]:
make_submit_button("a001")


<br>

### **a002. 簡易加法**
#### **題目說明：**
#### 請寫一個程式，讀入兩個數字，並求出它們的和。

```
Sample Input #1 : 5 10
Sample Output #1 : 15
```
```
Sample Input #2 : 1 2
Sample Output #2 : 3
```

In [ ]:
#start-a002


In [ ]:
make_submit_button("a002")


<br>

### **a003. 兩光法師占卜術**
#### **題目說明：**
#### 兩光法師時常替人占卜，由於他算得又快有便宜，因此生意源源不絕，時常大排長龍，他想算 得更快一點，因此找了你這位電腦高手幫他用電腦來加快算命的速度。
#### 他的占卜規則很簡單，規則是這樣的，輸入一個日期，然後依照下面的公式：

#### M=月
#### D=日
#### S=(M*2+D)%3

#### 得到 S 的值，再依照 S 的值從 0 到 2 分別給與 普通、吉、大吉 等三種不同的運勢

```
Sample Input #1 : 1 1
Sample Output #1 : 普通
```
```
Sample Input #2 : 1 2
Sample Output #2 : 吉
```

In [ ]:
#start-a003


In [ ]:
make_submit_button("a003")


<br>

### **a004. 文文的求婚**
#### **題目說明：**
#### 文文為即將出國的珊珊送行，由於珊珊不喜歡別人給文文的那個綽號，意思就是嘲笑文文不夠 聰明，但珊珊沒把握那個綽號是不是事實，所以珊珊決定考驗文文，於是告訴文文說，如果你能在 我回國之前回答我生日那年是不是閏年，則等她回國後就答應他的求婚。文文抓抓腦袋想不出來， 於是決定讓最擅長做運算的電腦來幫忙。

```
Sample Input #1 :
1977
1980

Sample Output #1 :
平年
閏年
```

#### Hint ：
* 西元年被4整除且不被100整除，或被400整除者即為閏年

In [ ]:
#start-a004


In [ ]:
make_submit_button("a004")


<br>